# 07 — Ministry of Agriculture Prices | Automatic Download (raw landing)

Downloads the Ministry of Agriculture (MA) market price reports published on the national open-data portal and lands them, unchanged, in
`/Volumes/rd_economic_intelligence/bronze/agriculture_raw/{retail_prices,wholesale_prices}/`.

| Step | Rule |
|---|---|
| Discover | datos.gob.do CKAN API, dataset `info` ("Informe Precios de Mercado Interdiario, 2017-2026", organization `ministerio-de-agricultura-ma`): resources whose name contains *Minorista* (retail) or *Mayorista* (wholesale), Excel and CSV. File names carry the covered period (`…Agosto-2017-Septiembre-2026…`), so URLs are read from the portal on every run. |
| Validate | Each Excel must open as a workbook (retail is published as legacy `.xls` although the portal labels it XLSX) with more than 1,000 rows; each CSV must be `;`-delimited. Otherwise the task fails and nothing is written. |
| Land | Shared helper `src/shared/acquisition.py` (identical content never landed twice; republished names kept as `<name>__<date>`). |
| Signal | Task value `downloaded`. |

**Status:** Development.

In [0]:
%pip install openpyxl==3.1.5 xlrd==2.0.1 --quiet

In [0]:
dbutils.library.restartPython()

In [0]:
import sys, io, json, importlib
import openpyxl, xlrd

nb = dbutils.notebook.entry_point.getDbutils().notebook().getContext().notebookPath().get()
sys.path.insert(0, "/Workspace" + nb.rsplit("/", 2)[0] + "/shared")
import acquisition as ACQ
importlib.reload(ACQ)

ROOT = "/Volumes/rd_economic_intelligence/bronze/agriculture_raw/"
PACKAGE = "info"
CHANNELS = {"minorista": "retail_prices/", "mayorista": "wholesale_prices/"}

package = json.loads(ACQ.http_get(ACQ.CKAN_API + PACKAGE, timeout=60))["result"]
targets = []
for r in package.get("resources", []):
    name, fmt, url = (r.get("name") or "").lower(), (r.get("format") or "").strip().lower(), r["url"]
    channel = next((c for c in CHANNELS if c in name), None)
    if channel and (fmt in ("xlsx", "xls", "csv")):
        targets.append((channel, fmt, url))
found = {(c, "csv" if f == "csv" else "excel") for c, f, _ in targets}
missing = [(c, k) for c in CHANNELS for k in ("excel", "csv") if (c, k) not in found]
if missing:
    raise RuntimeError(f"datos.gob.do package '{PACKAGE}' is missing resources: {missing}")

def excel_rows(name, data):
    if name.lower().endswith(".xls"):
        return xlrd.open_workbook(file_contents=data, on_demand=True).sheet_by_index(0).nrows
    return openpyxl.load_workbook(io.BytesIO(data), read_only=True).worksheets[0].max_row

results = {}
for channel, fmt, url in targets:
    name = ACQ.urllib.parse.unquote(url.rsplit("/", 1)[-1])
    data = ACQ.http_get(url, timeout=300)
    if fmt == "csv":
        if data[:2000].count(b";") < 5:
            raise ValueError(f"{name}: CSV is not ';'-delimited")
    elif excel_rows(name, data) <= 1000:
        raise ValueError(f"{name}: workbook has too few rows")
    results[f"{channel}/{name}"] = ACQ.land(data, name, ROOT + CHANNELS[channel])
downloaded = any(r["status"] == "DOWNLOADED" for r in results.values())
dbutils.jobs.taskValues.set(key="downloaded", value=downloaded)
dbutils.notebook.exit(json.dumps({"status": "DOWNLOADED" if downloaded else "SKIPPED", "portal_modified": package.get("metadata_modified"), "files": results}, ensure_ascii=False))